# Memory-first physical capacity experiments — v2

Select **T4 GPU**, then **Run all**. This notebook stages the frozen model/data and new checksum-verified optimization sources. It preserves v1 evidence and writes a new unique `Reversability/capacity-v2` run.

**Exactly one physical batch per update; accumulation1 everywhere**, including baseline29. Output-token chunks bound the vocabulary workspace; they do not split the Transformer batch. Optimize memory first, then find the highest verified physical batch. Physical50 is a milestone, not an assumed maximum or a forced pass. Speed improvement is optional.

Numerical gates independently test shared head chunking, midpoint force reuse, fused AdamW, custom tiled CE, fused SwiGLU, their kernel combination and its midpoint force-reuse combination. Only configurations passing their own gates are benchmarked. The gate uses the existing project optimizer policy (FP16 per-step coordinate1.2e-3 and whole-update relative L2.05); it does not reclassify the old failed GPU run.

Keep **10% reserved-memory headroom** during startup, training and checkpoint staging. The selected capacity path grows until failure, refines the boundary, then receives three fresh full-duration confirmations, with a CPU checkpoint on the last. A fit at the search ceiling1024 is only a lower bound; extend before calling it a maximum.

Final quality phase follows verified capacity: original50M training targets,1M validation targets, same model/data/optimizer/schedule and validation loss **≤5.56229485** (baseline5.46229485 + .10). This notebook runs short probes and does not start full training. Capacity, loss quality, speed and savings are separate conclusions.


In [ ]:
import importlib.metadata, subprocess, sys
# Match the recorded Torch/CUDA build and NumPy. Python differences are recorded below.
required = {"torch": "2.11.0+cu128", "numpy": "2.1.3"}
wrong = []
for name, version in required.items():
    try:
        if importlib.metadata.version(name) != version: wrong.append(name)
    except importlib.metadata.PackageNotFoundError:
        wrong.append(name)
if wrong:
    if "torch" in wrong:
        subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cu128",
                        "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    if "numpy" in wrong:
        subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3"], check=True)
    imported = [name for name in wrong if name in sys.modules]
    if imported:
        raise RuntimeError("Packages changed after import. Choose Runtime → Restart session, then run all cells again.")
print("Pinned packages installed. If Colab requests a restart, restart the session and rerun these cells.")


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, uuid, platform, torch, numpy as np

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
if torch.cuda.get_device_name(0) != "Tesla T4":
    raise RuntimeError("Select T4 for the current bottleneck comparison. Another GPU would be a separately declared experiment.")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
DRIVE_ROOT = Path("/content/drive/MyDrive/Reversability")
OUTPUT = DRIVE_ROOT / "capacity-v2" / RUN_ID
REPO = Path("/content/reversability-capacity-" + RUN_ID)
PINNED_COMMIT = "d071e1e2d8e4575d2fc9ca5ccb1ef4e3d5e185b7"
subprocess.run(["git", "clone", "https://github.com/LokeshJatangi/Reversability.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO, check=True)
manifest = json.loads((REPO / "results/recorded/reversible-v1/source-manifest.json").read_text())
for relative, expected in manifest.items():
    if hashlib.sha256((REPO / relative).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Frozen source differs: " + relative)
reference_config = REPO / "results/recorded/configs/baseline_colab.json"
reference_metrics = REPO / "results/recorded/runs/baseline_20m_fineweb_edu_50m_v1/metrics.jsonl"
startup = next(json.loads(line) for line in reference_metrics.read_text().splitlines()
               if json.loads(line).get("event") == "startup")
if hashlib.sha256(reference_config.read_bytes()).hexdigest() != startup["config_sha256"]:
    raise RuntimeError("Recorded baseline configuration checksum differs")
actual = {"torch": torch.__version__, "numpy": np.__version__, "cuda_runtime": torch.version.cuda,
          "python": platform.python_version(), "gpu": torch.cuda.get_device_name(0)}
expected = {"torch": startup["environment"]["torch"], "numpy": startup["environment"]["numpy"],
            "cuda_runtime": startup["environment"]["cuda_runtime"],
            "python": startup["environment"]["python"].split()[0],
            "gpu": startup["environment"]["accelerator"]["name"]}
mismatches = {key: {"recorded": expected[key], "actual": actual[key]}
              for key in actual if actual[key] != expected[key]}
if any(key != "python" for key in mismatches):
    raise RuntimeError("Runtime differs from the intended comparison: " + json.dumps(mismatches))
# Python patch/version differences are declared; these are new paired probes, not a rerun claim.
print("Runtime:", actual)
print("Declared differences:", mismatches)
print("New evidence destination:", OUTPUT)


## Stage the original dataset

The default points to the original Drive artifact root. The existing staging tool also checks restored `baseline-recovery-*` folders and recovery ZIPs. If it reports multiple candidates, set `ARTIFACTS` to the intended artifact root and rerun this cell. It verifies the original manifest, train/validation binaries and tokenizer hashes, and copies data to local runtime storage. It never downloads or prepares replacement training data.


In [ ]:
ARTIFACTS = DRIVE_ROOT / "artifacts"  # Change only if staging reports ambiguous candidates.
LOCAL_DATA = Path("/content/reversibility-capacity-data") / "data_fineweb_edu_gpt2_50m_v1"
sys.path.insert(0, str(REPO / "scripts"))
from stage_baseline_data import stage_data
receipt = stage_data(ARTIFACTS, LOCAL_DATA, DRIVE_ROOT)
if receipt["manifest_sha256"] != "71a12f5765cf37b7f5fb8753d51fa0828915a6db313ae20aef7fa9fb2e14df8a":
    raise RuntimeError("Dataset is not the frozen study stream")
print(json.dumps(receipt, indent=2))


In [ ]:
import base64, zlib, importlib.metadata
EMBEDDED = {'scripts/profile_training.py': {'sha256': '2251d956655aafbf86845e89ca543ef7d000ca09fb8236125c6da081c0fb4aba', 'compressed': 'eJzdPGlz3DaW3/UrEOaD2UqLumI7lqenxknsVKrGtsrHTtVqVSw2iVYz4jUEKanj0X/fdwAgeHRLTrJftqtsdRPAw8PDu/HAb785bFV9uEyLQ1nciGrTrMvidM/zvI+yiuqokaJJ87S4mouykvC7rEVTR7FUIioSEWVZGdPDJI2uilI1aaxeikLeyFrUUrW5xO5pARACALq3qstchOGqbdpahqFI86qsG4BVlE3UpGWh9vbMs/oKMFDS/I7LopF3TZYuGUoCyAFu0sAwv+eIsfy9LOzIVVvETVlmyjxYR2qNcPTP31RZmO951KwZfgXfoJMBf44N+nsNay9z8wtgyTv7A5dBZLBP2mVVl0Cx7snGfkVU7YqLNq82IlKiqGx7Wcfr3o+gKAJaEBAryrD3m729D+/ffxILwtEH6qYZ0HYWwAaU2Y30ZwEQUhaNuji+3IPJA1xakBZK1o1/NAeca58gHApP1bE3mzEFatxGlS7TLG02hg7LNs2SMC8TmVlyuP0C8yuTiFz3i2ESN4TLSMksLezeLaMmXodRMxeEu1pHJ0+fzUXdFkigMJdNBNsb7e29+fD+v1+/C9++evfrm9cfcc3e8+Po+GT19Pmzp/Hq9Pny+erpavnD86enydPjVXT0w8kPL46fRs+S5enxaSRPjiK5er6KXqyWJ/L4+2T1Q+Tt7e0lciVUBKRCyszFTZS1cna2J+BDNMXHs+C2ThsZIhP6yDJBAvulfOo8F2mRAIkXJ3MSituwiIrFmyhTcia+E97/FN7MzLMpYj+RN2lspkhXgn8HzaaSYgGLitsk8rgVP7zz+DDA4eu6LNLfpYECgOMsUkp8kFcoQjwOxO2dVI1MRCWjawV8WN+kN1LEa9hAFE7ZgKg64BJXiEH8pAChFEsJHfKovlYkwAgZVxGGINRNGPqwkau5xn8ukC9mHd7YGJg2+oEdYNfc/v3edXmrdN8ImPwGe19czuFfvx8uKtT6B1BfOE9xbfUNPTzasxirKK8ySfg6GALtHSR5A74ZbwB+aglKq7CP3Lmd/cmjO+DXvKw3HXa+M8Vsr4Nn0ZwebzpMD5+mAgz3J1rmHbpTEBxU+gBMw9xi2w1fgdqH3QLOd7drQLPy9sLroxIuN41U3qWea0ePSaT7UA1WW4EOO7gLIZj/6OxKoL/mURFdydpyTk1ipTm9iHI5wT/M18A4bC4HrLNJZZbsYqYxGDSyAyi3abPWrAIWBTVlDdo2LuskNBbB93TLoQdqZ4Dro7EhFbWd6bQkOZwMXLAQXzwmlHdGE8/F9LaeiSPTNNgcaLnfKZgTmtGyAxjeupnkMle4HiWYA5gTPDYGuVtW8eOMIeUbEgn0cHQb1A6Ss2wFUVXJIkH27tqberOL2VYpuAnZoMv2Dd6+yUNcqrIatH7FllkSw3L+uk3rQf2Ltu3PbJ0lGRq13ubt7e3QO6hzHI/YJ2drrtWQmlnz/kmi9xTVG3FbI/BaibKArRaVXpZo1lKUdXqFPCDA//pdFmRys8NlFF/fRnUCbnUirWVvwa8le2t1H0L2y9tC1nMt1Vm0lJmjVSz8hbgCP60BZ9Lp3xEDHobYmi7bBo06tqLtoM5BGCZpDP6E7f4P67UHiIPyzTwdRIdOsIP7EDCA57C/j+u6UgO9B0zbRllIyMPk9NefIcvGwFgR+Kc+L0xI8Nm4fax89RYE2iK4QCf0LGtVS6ARho6Ij+g271bWXzBsLbAjCObZUPZ6BB5jMznJmKj4QRKMAQDbbN9d5BzL4YwiN/bUE7oMSxDja9x4YsSAfqqBoUOmo4YAZizAXMBA5FYPvsYlcH0BvvYhtMEf2IjQNM+2gcmzahsUaJoYT2MZw6a8lkUo86VMEoqEp+EMuk3DqkqVEsYPghv3nIZI+j0syjrfBqnrMQ0hy8O1jJJtw8u2qVpQeXX5myQfw4ECXGdkXispzygWChEB0juAMiMzgK5KAbFmGzdDxwaR6SJFgJIW+EOGVZTWiE7XeGiaHHB9pKYh6rUZiFmUL5PorAcZ7UtGcHNYsfRGEoDrZVuQqvCqjpJQFqg6El+rDRca+GVyF2MyoKhtSoQEGOKfwVoZo5vfKgdAZ95Hpp08chY+FDAGBNihcM6GwYxWIzoqLSt0XH+XddhWmEkx+2mfQ0wGuOBfCuHnmI9ZpcC+cVursu5CwL6ZosgeFC73Bqd8vVEpLopDfgWgPbY2mSyuGrerkv9uIfiUIbfoblEct3mbkWl0OiPxUmDX0G0H2ywrpQfGZVs0MIIx2jfz7fcg8hR5Bf18wyLdRsWRanx2Z2C96N8svnir6viZZ3qtsjJqjp/Ne7T2liuny1L3ub+wNAGWSxUy8eXlbLY1HUAZt4kxFK+uqtMTj5nQcSyKNsv0T8Joyoh5vOGHsMclMbXnGnez/YFt9tEBakpQKIVcfKpbreezUil43JAjcBQc0UOd+NKhuGHRkJgzKq6k79LemXYXnpgICoEUIfBhoVay9gasfTcXG8rhpInZbrCLvmFbYlcIjixuc+4z1xzhdCGWmY2AA9Q7sAom/QJPnF/9NQArDS31rrWR0HVKY2yDs/IqbRRGudjTv5s9GnRc4waBhNRltZkGrRDwm6DX0+cZgbHJ2mK68JhWbB7M5mM1Of1JrwpUiZgou1scHB8doa5IWH0vPNXm3tjzRqsFG/k7cRBheMi78uh1G3s0tWQMWEipiVSRoRp36SMRGGj+GNVppwk/PEdAf/wO2GwK2rfiLekn9NyJY2X9RBGEqBYgxnVE9DrM0hVnv5dyheoe+xfQLvIU9w/0XvAoaSKxZfBDCjkS/R0THwKcJorX/oz1HPxNG5k72HfyDiNIAh9UO21BdCGJjrO08gbJFbs/mIwc75GmrYHiW4XVIYUUt8FfUQRtk0JQgXOxDcf20PhnUQ1ubQNmExi7Z1wYucvZgwtS8Rq4Wq/IouMuC6h0BfYXpc0f6Bpib54UE4GcLW8gWpCNsWM0f1TnLRqpPC3842C+Cw737YB0hInBxSykY0QBGhjbqMbjmhAPf8C4fCeCp2AkfdNn2H6wa/QMRx4DDDxWgTAXYln8UqXw3FBh1iGU1XpJ2ybb1yufa9z7ycirugSiYEhprRbtZ0gNgxiDnsEMNWUGsnqa6cYM18FGIXN4fyz/WjQXhkmB/iErgdkUDxO8Cf512Zx9s3HaZTSB+JuefiIsjVLwEt6gBANpz8u0aF7XdVn73qu356IEnxGk+xazCJhJhC6CpwXagya6TiHIS16CVIHOARpercFXFnEWpUZ/67D3i4cqA9weR49o5a2dXfLXKC45IyG16gXaDbuedTrlXruqIF/xtXFT8ZAxa7qDFNQSxGOpWuEJhelxwdgARwKj0LlGWWa+ceYHXfvIgU/WEXE78d4hzyIYWvChgYH82FESvi2lPQrqZZB2HQlNnEhYKm9JsiLnbs2zYuP93mPAPHSyYXygvsM7PesDpxwalNlmCF5gB+ry1qdvmjAWX0w/AExqC67lhmJW4C37jDmtj1dctcyJYau6jvgQjxmpBeBgyi7Ep04n+8zp2YfN6LvgTWhMEOZdDwsBooijYEg7mkl3nQTjdvgKWJrcYOXAfzZbsgu2HtCq6AoD+aMh8LTAvIBaR5Xs6OQ+NDt5W9bXsvadrBzrd9CMdI4KwpQoag64AUwqGFw6aNXWQT83sals1mWyoBH8fS4mAkvuwM79hGNGgUSS1gs8/6au5gmZ/rxKM6mPcI1MOjNSXJanSYUqwJHLPqa9bMiilwLBc3dZMaLB0/4UyGgqL6/lEG4/8DNxn87W4q5ZR4d/+sjy3hiuTleQTplNaZx+1IkK05HeVIXRTZRy2nSkHD/wqb1Wiz99/vmV+OX8M0juv9sUHP5AHBwAHgeMB9jYKmvzJSpHchg5sMc8dqAxI8w5CxXk17A9vi5noOhzDl5xqpqwvHb3ijU5HUqhdiU58uq2QGcCcyyUnDrT+wnfOeOVw1rhKRV0BL/BxvpYLQGdbkZRDp7zSQCdV2HbxDDIlJ8EBWgsU4ECvmYMTrIqV+j1N+NgyeNthfEmnaKfhE4hBLQ6v1xBGcOTxU1al0UuSRMOCyi6kNWzrBAirQ0t7FO2DlQT4dAfK0SYtAFKrjfXhJ5I9WJmfYJlkZPAIvqcNpnr3MhcM/IwSUX89F9YYKG56XOh2gpLRrCswYKdTc06Sh+Jv3euKjr73iU72aOODyHxE2dUgPFiKRMlyEtFhHTKVtiUrU4p9BFEEgykcVtmB6WQCLRFCrEttBQZZhrGmP/45viZaB0aArLNGmQQBNRBEuQAwksQKqx7ASHqKUfkAdOBuWA0rq/Ye9DGqr0T1wsL1zA+qjiX9XugHkPU7YDBpRoUEz1EvHdIfgiz9TmaatpkQ0ZEycYu3aHiNuJRUBeAxvPcBRBxUFdBVI14gllAHO2SLzxTSNeFchBYHWOIdYJOrUsnIg8O70Zjs0KWJzD4xZDioXV/0vMKQBkPGxG1Qyyhg1BN5WhdnUUTdDzdq/CINY8qXVLF2VJ42oK5PH7GlT9gEJ2hlKw1BGP7qTqjO6ivYANGTVQqEW4fQJUU3P+4w5NpGFY2421mNvnh7Xpk/yuyznp3J6aDyJjm07H8d7z+GWgpjIB3bft8V47gQR1wjiEI+GRtlmgNRlxtDbrhbzOvwd3ZKa5/BJ8YdE5HJpnA7C9x53e2swaD9eHR6VQXl26PKCXoRRUWKIYoWwGT+sfccFfLqPvOnExuQBTwx1qqSxJxwVE/cxRS4DExTJVtHUtXtX1BzxPIhQer6Q068lSHCfFmX5pnlN7A1EZv5Re2aDOu06pRh7r2J7Qlt9UGTOuwV78Ek/vsK7Ygbh3oYa+u05sFV1m59L19HDGbXXalOjZnYZ1PehK8SqL8X5OJtaxebEnxTDnqS/Be1MLuJv48dqUAH5zA/s7FrUyv1g0EMHG0sTO4D10u0FkegzO4csEvIM8f6bE+5vkDpzEL8srRZpPKwVC7MwkZa0ddqOmbMzOj1ZzcHOfzzLHJHxKHbbUqw5OK4ZlMp5UGyqSXefnrTgyJKrN+UsvkMxcTqrPbQaeUdiRuvAKWM52bVcbNNVlEJ81kkNrfn6LV/f/9LjDeqiu/0bNNmr/+bElbc6lQf/DE3rKNmSgOcSj5UlDdGwoGxjNA9xXrNJCK8bEHngBA17+aGx7A0K7Y1JxMoCoOeCFjWD0+xgUARE1+Aw+f2pzhGMBjuNNlRr1tLjOCmQISJMh7djXIkZQ/6tpUm/u2feKwzctlVLj97QWEAFvcsdg3Sbf3xrbdc6kGdmB6ODU5o5F1wWNyn/xdHLNOPAomYLuU5BloAvQJ96f8p8NHkUa1NTgDlFkIO2HfDvZhBQBWc9rFHBWcpTcQDkoSykHV7rn+8oo7bYKfzj/35fZxaqY/kZWFhyb7/POrQbUr26NJy0Rrmzhn3lGhOB9WKesvfofrovs6fYCsC5s5k6gzPca/4Q3SDxEZ3Kv4mqsR8NaJ7jgm1iPVIe3AX63YDHUmZ/vTKomAPFYt4WfSeOJHEy+Qd5gkCPF6SE4+ZYwEq4f5IGrgRMDAhlNGmNjfyeWzLyu7YxGJ2fswAlczupLKp3O5cLkJnUwyb2xfQKZSU+aKmjLZKcZgyOuskvV2kE6+kB1WGmsQQHnBpwyXAdkRhZzmFNgPEUJiwVIwK7slRb/dmeR07MQhQ78IbfuCm7vG691Q+kriBg0lc2kRy83CLGaO9dRhBpzeLJ4ezaYpyQH4OlIhLiZkAhJdo2Ljy8AQQi+5X/gW/Eytn6CRFNOYPRieP5j8wXz1A9htOxedyF5fy7qADYLoQt+JiqMK7y8mLw2O9eFPn88//YrZ7LawkbQ3Ta6UTgAwQwuWiVUZx5G7vMPx6fIOna2BDqdHhYoB/UJQprzvsA5vEeyi8bTCHFXXB6FW4npl6xRrDjc+Zi2wvAjNwPERfqbVmYsxMumo00N26P+FYu87vqboZPJ6ht4zS7dH7xPeZAxVEVVqXTZTSt5e9LW9giqNr5HH+yhMXjf5w3hN848uul1QafGkiNmj5U7F22qd8lbtGtPdgaSYTbvsW07HDdDBNbzdap5O5icpsOuz5Uy9h4G9xvcwAp0/q89zNIfyadmC6r8z2eDBLzImSKpm2rmwXrUOqhfkRfVX1HO4HS9kMeVxy9VK0p0ic3arM65qMZmQ3Zl07aMBmkEuvDeA+ppLL+ZYtyLxyi6NOiTppCNc8evh+5fGb6Lbr8rcgKeLy4YgiVi1WYYnayoArQS2BU0kj0iLOGsV3llFI4SVNFlUBeItcZbeKn0bF7tzTUxG/Zl3oH3OFsaU3x/gbYuEnXw+GqWdD7xxwoMrNcH6F6CdNN+ShQc2uXCOwrdntufjA7IbQGupy7QHxKUsl2XEr86dT06BhxnDGgZwwdhTCfGM8K+f8HuWBcyFV414TX+Q0BBHSPQCzh4QlRWYe4mV+9Qb5KSqffrqqMavO0iludAb+QOHsFUNzOzeT3dOwG0mmh/AfnsMkkLintKfTRyS38PDFTD4Wpd96/vsLZZOOcUdf+S8XsWg1Ou0pGjB90xyOjx5QWfE+id8P3kBSPi28EJ3sHUY4w6np4MOp6facd9RbLEVnSE2A2SGuAxQGWKiEWEr5yTv0AXm+1mmuoX5HG9aG8z6tz4oFUHn6jrTZCN3xx0cJA0lGLnC3KPo8xfdzTvs387Hj66RQEyxMkLeybilwAFmOWg9fpXEtrdQIEsdHOiCh3nHcFtKGQT2xnPTg4Ru7ExV6UAPzcDc7qyJW5l+niXk1BxEWw2AvuuhfH9JV4ccHOh8soOIzpFPAjUJZ6e7fsTQaXOmevVOM2ejpNA2jh3skHEtPafWZhAPOLTS7wfZLa3dVQ/zYVWz8s5teeMXZJz7M/GF1IXwV8byYgXnzOurDxcUefL+GCVgwpXH0AKwbxDiQshb+N6tR3keeDRFAnrFRPfelQCMta8JAybuNlngKRfSPAHuWQAQ+g46e+EM+vjp5/efPyH/YsVhPBQF7ZrqQ35XmAbqeWJUvz7CgdSrjuhcZm6jnVDmqplb2GQtkEd2Bx+9K3XZnVb6L0E7qArcLKSZdz+BlA1HALJWMt1Nfm0KtDB0JKHCJf0MpyEjg9DuB/XLEzaMDIcxYVvqtB6oUprzrcI6UXZidT/NpEJ84Q4XT3imJ5fIqM7G3U/wp/H6UENmmV9fuGuns74jrjO5MAS47Mwr+93GkebbeXwvjxCdbbXuWynTAaNCJ40bJ49AYJo0ykJkBnDP/xz10L6RQfK+Fab+44B43dYy4wT476fzz4ILbnASKgynrAmodfKGMTrZXoFIXck1Vy9tycEBQtIvnlLomGUtiHTntvPZa891Dww6/xEfNfeK/2DdQ5oYp/5QwZO3dMphKtr1SQY8/9iA5G2cl7r8kv7YPbaBFT2laQ4ODvDf2fA/rzPh7mb3CuNqdkb11T1T0YIjJtnoG5eNBieyZQHs66RG8NiuB8WcPNkeV+mSLsDj5UZxd4b69Q6Y7mR///ToLDhd3dNm3blXbvReuaktC444xaiOFZD/Sw0SprcAZAyI+KW5eDI+3XlyeRYcr0x7/2QK277nNkDXh/apAPzJ5WzUpR8gcw/P5envkKlxB+meFWwpmfyDkxeaC2HdZUbvPVtL7TWQpMFyUGWLH8kTOz0VLQTFXeiqPbRnz8Bxh5E1DAc2e/oDsSqQMurCSffaBLk++gjLsvC33+rAEsey8wac1Mi6qiVnukzP9+adbJg5PtAVwHQtrI4KiDJt5hLMK6gXEXFVYsHvherCVZNFEDsu1+uCh4lr8wwooRw6/saXYdHNY12Cx/epWXLxbQsKqahfP2EwrGWOpRE3Kb85LC343RW9d869NBPw6Y0SSakzvfTCC1AEAFxYxA/f/vNc2GsYdEBs6PbgdXLMevQug9NS7HXsXn4AVBW+HwsIimQGzE3RdyB+tRSmcmGTEmB3x9LSLIe2u1WNfu8Wnm/mWC/N+V3DCZw/MCTHFXPEr3QmAYsUI17vS0MhgIT0zFHHxDQEf3RJBVa4YIpQ4VoqfYYdbAs3Xa11sslhKNqkWq5kjUF3AOoSln5Iwbpz6QW5FqDw9UWNbTdKkWviTEGszzPEIENL5Aa8bYrlqrVA84aLjLD+9yYtWwWYr9KmoQpFFMJAvLKvJTTZSiZu72Vn1RriuANDVJZ93NYuAcipGL1k9z1omj6/ao9q9IYEcDLdO7RzfW33gAgDQhSBH3cIzw5UuWry6E4sW9AjtZpb1hG/vH77Vln56RjTXCqL1yXegLuCLiZjzUpEvALdDroW9zlumWz61rCmfJShSk9KyQoe0Aa2SRt4kqMnJJVWfZzlo50wS155H9E1EWyawH16YuzSk7FH8mTgkXQuKPurHLQCbmrevbFGtTF636s202RVT+4DMEKcdbM1ieAiitK8YrLnC+AlNQCcrlIQHW32hv7Vh9fn7z98CvKkf2SGb+njsn8yFd2L+7TZNok5XOLuSw8fy9wVgCrCDKpe9ku7WFOerLRyuVJ0eZHRhEgMSMXHOJxoobdhhrgUU+hdceBODcGr+opE6Bx/UShM9X54uToMkzKGWJwHBWAFEAz19t2QnGp0z6le19zWcHzh8UAnOv/aoTZs/9qB5BvDuMjcG8fXWIQgHia+HQ+xSQAQmhTTyhduDqfLyFzikc4qgh1ZdB22ATVZA8I/xctTZuzJi62IcEqhQ2N7ssbFpVccOIZqExNjVJ5uG9MlHibwP9o2api0GI893spjNgsxuXVzsZZZtfA+pcVGYEQxcSnIvFcWLxOB5k/QeJgrQiAIVeAKCD8O7EWzqLtaptNRuoetzF84P0a9tFAu7NdBD9AOWLUd2crCyKR78Gsvp4MPOM0k/iaOnVvhgWTl8S8NQscoVM/GKoJ9THIRwCrydZMbQ4PJO3VbL9RphB9xFLDzuGXbGswVGXpJsPvGli3I23s7eIfZ9/Ttu96l+Q40tSWOftWhimYVmXRAo8F9EH90GKDrv+l60LPvyUHacYHo2feTKNkQWKsw9PcbZGVzZFyYuyNoxSjm5Vl7N6YjUPQpvloVky5hSOmDMES/OAy9LreOvD7kdCcJb0Ntesam2L2Bufe/QPXSNg=='}, 'scripts/benchmark_optimized.py': {'sha256': '47d2bd1ed5d65f98fc835c9df45bae2ac3cba76d48d361bd8f985ae03e2094b9', 'compressed': 'eJzFfWtz20a24Hf/ChipuQZsEKJoyxPTQao8jj2TWsd22fJs7dWqMCAJSrgiAQYPyRpF/33Po59Ag5IzmR1WYuHRj9Onz7tPN757eNA19cGiKA/y8tLbXbfnVfn0ge/7r6u6zpdtmTfN5Cxr85XXnFd161W7ttgW/8zaoiq9/Osur4ttXrbNS6+sWi/z1t1mM2nrrCiL8syruzKGxh4U2x1WzuqzXVY3ubxfVrvrB+u62nor6AIazj3xRt5HHv77z6pUdf6nqUp5vc3ac66/g6tNsZDVP+ILcV1n5arayruqkVfNedcWG3XXwpCatljq991iV1dLQIB6cq0uESp1XWfLfJEtL+SDstvurr2s8cqdKlPVy3PrJi7LeN2VS0RktsHSbx98+vDh2EsI+iBN18UmT9MwrvOm2lzmQRgD8hDXJ4enDwCWGAcdF2WT120wjWAIdUAtHHh+Uy/9MGTc1PllXjfFotgU7bXE0KIrNqt0W63yjaNULKYZpl1O3XaXLquyzb+2kbc878qLfJXioK+yegVPdl3alNkOiKQVM1JXNABFC6Kht58+/Peb9+kvr97//PbN52MAO7uEaW6uy2XkbfNtVV+nOBlN5H3KzwA5cCGhqdNuh5TBPVDLAEOTb4pSUc4ia5fnaQZQUvfNeTY7eh4hJeKcpdu8zaCJLPKyttoWyxS75/Yus02BracSFRvV6IePxz//8vN/v/mUHn949+bTq/ev33yOvC8ff3p1/Cb99Obdq+Of//4mfTd78OBvb179BJPwC0xjfDj1vO+8t8VXQCNMYl5f5qsJDxFGvc5hNrbAEkUZeSV26bVdCUXbCh5f5MBMNBag7aaJH/zt58/HHz79/PrVu/T442do/uj54Ys/xy8On09fvDg6ejaDzh+s8rXX5PkqgKlaF2eRt8ovi2Uezh948GNOiI0CJz7e+KchsO8u3vueiXablV22SV1FqItiLbqM2+td7iWJ5y+7VeYzAPjjdvCh2ViabTaDBnk8evLXVR0QxQK9ifGtuyZfJW+zTaMGmbddXYpuqG78apVt/zfXRBbKgAZgfoMw8jZ1Ijvd5FmNdJrWQAL+aaQAXgDFNIkCDm8P4b1nPpgBuJF3lRdn5226ypfZtWrXfIjVGGL6V44QyLCumjbfpTTheSM6EyPanV83xRJkRKL6lI+4QtoAevxTOQGq/A/eoVfVolJ8lreBf1ZnqwJkSJotl92225AcT7Hvxo+8w9B7mAwr5es1qILiMje7i1Q/VEne6IkG7mxy7+/Zpsvf1DVMnf8622VLlEFab8B0/doVwBweiHi4AagV+NQZ8KkB6aEfmpN8IsueCky2wE5526Q7JSpsVIp6TbcNRpEeeo89TYm/dnm5zNNNXp615/5pjyZXshdBlopWQaABWPiXpJQm2GVXN1UtGVOIUhBPORAE0GPVNCRV83KV+ETDvgBdNR3D/1WKExk0eZu2VVoC7pLjussZOcsKZB0QyzgyHjAjtkRTzCnYaBMgTzBkiZAcTIGsB0E2Jd6UngArMtky9ryivIuK8fc18q4jlrPQkpTUgUQRocZ7oruLzD6i8Vkx6tLgQ6tH6Opr3FaBGBE8Me4MuYToeJIMlJucWwae2lfTJidZ8SXYTEUDhIowOeZUd6dRCl0SRhT7qjcPE+5uH099rKtF7m27Bq2psgVEeu15DpO43ICxsEJbZAGapVoLtAui8JW45iF4RUMG3HsgJd0dv4u7ki7SQNEg1y6reqsICMwZtKeaeLkpdkSeKb5PnXJX4ouKYQWpP2DwZyANGmg26M0pmDWy2jb7ynZFKofDwg+ma9vtoO62KIPDONrXBpfVDUjeadCW0JIWWkp7qgFai49ASAQjqsOb7KuN4iU4hDbQeI2hv4AudgU8l6MPQ8VkZ3UFI4JZ1fxPqEzpRaPniu4BlhoAYASMgfdY4CkSo3WQgk0Gumtk74DL56Bye5QC/QqKAZSmdBmEfWKiJjQhvVRExhJKl1cAma2BTqMr3bNmi7ebCpREefaxKspWsMerXz56FZhW6011hS6K1+zAuPCAOYqt4IHvvM9QCaBHRgXVhOY8eTcgNwFD290GyNZj8KgJ4v3JBiy2jRCWZL3Gpoa58bE1f85iJS7afBsw9ZH40HoYeQSK4Z94BZbE8hz5w5dkOdfi4FbonqX2zNIzLdS1SjFsIrdGAbl1VZxtOlmGBJVlRIHPdnwFyCLETJRkF1hovOwMmA9EDtjN/8xLJFSUlAev30RAqstNt0KDvy0A02z+NAfbrLkAK5Za/wySgBxF77JaZgvQ7fX1wSrftecvvddffnrl/eX7A+FrHB3OvDPuEoovwUAlE53noGGUv/74xYMBNN5fZh4U8BaA4xVOMjfx/Bkx0qZCmyL/mi87nF2uyrb524+Hzz05I967GUwakF0GGsab/YlRaDyKp9PZS80TLLbyekIIWlZVDYNHEK0ahwza1Tk8nDAWAe1o1Fzm2GM8PWIuLM/BMlKIvZzF3mth/UBBJUS9VQE2WY1KEDBTY1vop6CXnmOhNt9cixF+fDrzNgBq28ytMQJQ0+mRGB3d4LAEaHoYPLxqUyyv948AoRDQAxjkUIlBiNowFklbQtQi2ZKo3V0D7ec7vLBMFKVOTdNXq1gqA44pvNUlQWn6693TmU/gOp0Rqsc4gao3mhnleNLNDDgvns5QCGEHKOy8o3xyFFlyBwZD/CXrUaVerVk+eT6opX0aMphQn2UL8PK7FttweZsnMKjD577VNI/z9MQHV3ZjuiyDXtTAeOJ4fEP3dU8nt0K6gfRZIdZOTrUdmLfn1Qp1VOBLbxxkDCi/1Q5FsW9YgKTSEAJiYKoTPP1zFDodx13nMwzB4XT2LPJm02ffR96z6YvnYWgrAOGtSGOcAEr4T+RJQ46iHYmPYwDZVXfLFiUh4h8dmiQ+ikjcFxspC60uXG61VaCqC3DlyaQ2giuSnl0mJ0GuRFqfEWR7dnnUiiS8573Z9jxHGOcir0vAoAxj5GW2wJgINTCob70NFGB2/5t8jb6F7ZRLUKVhZ1epUQMM6qj2e868XVeGT9i+kBGahu5Rx1wVMMeddCNJ64DeKkD+70DhAU9vQGytsh3aBcKcEFpDzauwXA5n38cW4R/a5cBkAEFJ0HweWiD4o+ALmSnAA2uWrOFcWgQkZx9zAasqsoSQp8AOoDHO8mAWDptHzO/z/V4you90D62pyYUWSQm8ezmDFlLkJP7O+sqN1EM/dAwdfyQ12NlMvNl+efH9eAvsMgoL+fmzPU7lHTJpPETg6rtYaexgmA3EorLPSU6gGCdrSARzwHfRI45M2O/GKlFLtkDGT7DnuK42m2ByiAEecM+AHIJ9lU7mkTd5Okc3YnI4nTpLAmtVWzQQYAKhDxhNIHp8yLXCGIMsobsf5FozohxI01Wp8REiICCZ0t7GaJQ2KfB4bYrLAAYcxmvQd21eBtPoEMN8BJp6CE+KM7C48xQFxdcE4UUreNWR4Z/4ALrvhhx/gcnKYNMbuAhjFTVw1+8xHPj++FdZ/gAi+DAjdXvM5gpVGEIV0CAHHpkgRob+7VsM+qflHGJLTYv0FehfMrCE8kfeUPq+53W4AyDyRyGAHL01w6wwfygjMtDiKCL+WezURPdDCnLo1vMROspi7DZeFZdpQNMI0nMxeObm5DUFSpiVi2ZdlGDwBtxgGGMYO8T4qaPMwiwzTt/jzux7FBjYlOkD0uqcsuxHyNZAcpztUDUGTGkCcG/iSejQEyVXVeCI70FsIL9SXCWfHM5CB2ffEQcam7exUNA3NO2e+3u2vciBwtACONlpPhRSkoiPwy8u6IfEqnn0fs26IR+2S8qfoy9SzxuxGPOnPcO0+RUjMiCGYZYlCJOFugzjVdUtkMPj5tcO/LdACG0ZrmDOu4PxRvhuCJhwPZxAVZvVPaGBkvvhYbw7+q+rK/L2WGKB/yP9A58hgyd8AU9IOoLtBCILYzB3SUxfq2pZWuhtfiPUnHzHenysKbczCi5iYHBxOFK7744KLrfVzsTWJCCRFoC5A7sQPxzy/UjHag76ziwCLmXNZBFqXuD28XX4L5LaCEj7nWwDrh4lplk4CRb2o0XoBjii8vTv7wJekGvUlxv3GJTTp6dYcvNrDSaVKQcOEGrFgBFM5dNpGN66eOTEx/VmXIAFbkFVhc8uTr0fEu+SRn0RXeJIOX5CzNm47DwRKpDqBlqxy4DN6xl+ozZcUNRFLOMiIZyjgVzl8BAAinFtDgaiqOWoAkKp4gsNxcWCm/rE5upTKlaT48Gg3pLFT0M2kNAvhWUGbSUJ3QvBrWPAhCOYFr4AcSAa8eeyucjz7WH4c/seCgAIS8xKAJvLn4tVJQng/E6Ae5Tkk5edKjpCIdHRWptvGmr+3LyLZDUODvhz/otg4PoT62IymqA5+huZpMoRQHjlc0hwcjmbyJDpRMcaJzDJEzPAOHk381XY+zxfXpB9mYITtmfBtb/Sus3KYp03bXqeNSCRVwVaTlV93UuOwBC+FaFp2qymZdRiCzyb1+uUgvd5LZTuLrsG8YELmWbyS3Aj/DngRVr4wlyWPF0Vy5YC+ybezXUVo5CaLp+HAiXlmMQTkdUCL9KUQztpGvg4vk2x8EFv0usAU6XiVbfdyaXYCJkhvcivG44HxCAeAEg0Bc7zr6viDNBkAaBwpzrsYRMA2gJpA4eleslCYF43w6/SRbeCP2o4znU8RVJVuUGKQTiNlpBiVA/ewcGeRW6jFue1pXWJuBRZLlirpRUn6JIytsR7nQhD606qDNmgogzbo/ge7lUZY+K6VWYVpXwXq3zKrsJYwgwHGXAdzmhWsxmSD74cLttxRbXYpunqNnTTOQYb0waDohyjGJI7eAfEDPdkCwqvKDbDbDSa0Ynm4AlwTLxrORBv5GEFgqciaoXbWxXNxTeAVwNtVjVlKjDmsb0Am0NJsEtxGYidfAzlyJydhsjNDPtSNgnIdrC1gktcbzc8NsA5eMMldArald9GordjqhLOvesiB0uV3ql6+cZZE6en5w+iDEfNimKcSsX0Lyha2TJnwAkAsWh4VzdB2+1ALm+Kpu27xYPu7uglQ398ASjGxgL5XkxeiAEX87mcEmEkXOZ1sS5ohsAQDrIsTBK8WCxCiiggX6BBppgiSRb2LUfhsCwjPQd/AepEi4EdCYBCs2qBG51y2f1cONqfODFQpih9/GJoGrCHunLV1sXOW2fFBtRraCt4pCvUeG1NJIZiQklKI/1QvVxctzkt6pKZ1mKoJ0bpCkrAEvvAkW2Vkl5hyiehqfkUmiK+uMLVNqOMySxGg2oYqRw9lJWX0Bav2YIMR0ks8jVT1E0kwcQq8ePHZnamFCFSPV9V9QUwI0jjRhAXXsZV1+66Nt5egDwIRPoq6Z7Iy78CkaTVhc11tJwueZdvA55vtM2wScLKFgwvsT5F4pKrn8GrbVUWHJVEaUiPxSNKqDWfA2F2m5acQhxTBxj0667E0eP6lepI6CK7d6H7MjSVMBkX3l4OVuCQsqBp8KC6dunPZVZzXFZXgUxsjuEVOLlNBbS7zdhKOM+zVV1VW38u80nZVichaSAWhSuPIkZl70diTCLDq762hFaPiUjNIBfpWBWpqaJJs0sg94y88RE+wWV6kbu3MoJOrHsTsjxQGDQMrVifq2FYKQV8DYfhO++jyOIDv3SQIDiH62yJyzio23QmAiZnKPtJOO56wUa7snLF4O7MSQ1+bx2RRiCCBY7q/J79fWv9MNq38GitO47EAvckayaHwECOtEwDmP6kYDaSlaFHFRqRLjSSu2eSz13NPAQHTHd/6krRMTPXjs+LZjQTtJ8EWlZWHqhvAUY7D4jMyOmTJE2PKDpKakVBhoD2COQuWL+UOiFeVeWsOwyEY8adWv61QZ+9AMVblNeUpaKqzkJ7BLZombuSHRJeiDdInNjesnkT5fsY3mmiURN5hivJz4U72XMK+R37hSPEaYX4qbjtMJr+Ir+XPiMoeDTCKtGLujXepHl5WdRViYSRVE0sbjkd+eP/Of7w6fXf0lfv3n14nb7+8P6tH42VQTllFhyLq+Dqt+6xv1lAp44K/8vQ7Ymp5w1hZ0g4nqoTv6k6xL6wEJDvbshyiFUop61oBwfYZ5b1oAPGFvAn9kaRyMO6B36zBE3fNgf9HRjx7tqP7CL2Fgoq8FgETAJRsl4eWDkF4F6ebapF4D/G4mF4ejsgZOS5HkcJDJxRBmIylsImlgEN4hsSlklKmoIHSRIoAOxedYjErdLed1sQRci4JDPY4HvpNVc5qJsFEOaFpens/Ef8KdcYrTumbSQecL/qA+VJs5521jO1ptWWpTdf2i64RX5WLSdKehYUreCZzT1Mevtz5g5p+JbzumRF0B7NlmLfdpcuNBDBxYuihwJMex6axA8TNaITX9ulMlTw5DB8PMO1toGl/TDR9fBlg7NPDeCF5D/X0I7lbiUAMQPndWRokrL68ZHTxMKmVYf6T8odmBJgNu7YJ12RMQYPO3A3Dp9HwlLodUeubnJHPr+BSyr/mKUR5/o+oWuRvhn+iPH8faiN9kRoQhfefi6bbr0ulpRdKPL+7FRz+TPzqFwJDBQ1S/ZnUMUEUuCeEx1ex+mwk885aBAMEGYIlX93XhVB1MscU2rXtddJ4EoLxR4+KdKTsP0Onkb8V7AaaeUepoV9o322v8vSSEQaoApB9fBMu/ISsTtP5m74wl/sR/nXuPVO5OE7Vr2V8YP2iXtRHOMOyciOGx3/FeFf3ksicCYCkaZaYTvIMH0GGma4lDEU9U7Q6t8PmsDpsGshtmlZRyUSYK8nnGB+OpZswGXsNHPNuvuzC7Dlgz0ZBSL8gV30GclBZlYIFLdBgnrKs4vUFU24MzHOEGk4GKAsLvG4t+8Hf4NIp4Zf2GMYNex2AhKUFneB9AePcA9f8u7k+Oq8WJ4HPtimqyKbNNvCd/CQGd9IzPDAgX+26ybYkNixHS+bSzDgql1eBv6VY2qNCEqiN0DHH6nGiQlH5E8mv3Z5fT2BOonZB9xH5j2jIeI/MUowqszhjgQgisoKAx7AIiXMIq6Y4ftNVe0m2yY5mk5BITXtCsaUmEPFZ3ltwfnTm7+///LuXU8GdTUB0iQnpy+JuHO8vB+tSZU5xLpJXi85Ap2MBcWtSUeZgZRr6WfuJ3TR8bAzNSK5uurod0IQhXLAsqQlO5xcIeT4aXLjyxVjGV9U/UY+t8qrhDncm8YJlxcBM4FAHtgBZnSoVhy+mL/NszLt96s358f43mwBKzRdDXqBkoP1KtSw82gsgtljSTOK6NwW52CWGHBOCXY5JuiYL66yoqX5QfI9nKL9btAwZgQ1rnRMkxGxd7taMtDKRCQZblYa0Db+aAcM7ZIvvftICMPjiJsd2DxYuRlLW8NAo+xdEtoJJ1d8BWMefE2ZMvGVUwfA16RWA+D00JGThb/86zLftYaVOacFf4c2El4eYMwYCBEwwYSW7RYoAgRJ5Asogab4ImKKw8qLrrlG+l2i1hvQ3NeT6akegqhNppW4dq3YmT8fCCn35/7rCo/AEIJ/gp2qFoCueePUFo8BEFb5S3EqgmcMDukSk4owsTevPdywdgl2acQ+HkdRcAeUjKcC2J9/8arlsttl5fLav3VyvnOf92myf7e02y9yhwiTu+OMZisWRxtuD0UQgKGdlaoalIJajk5JlEOFk2l0l1d0eh9VP5ha2uCX9FZ5uWYKagmgbou80Z4LFheWgYuU8ZCIBLN1xowUMNXRtpDnSaS8pHQa9cX3SDmXNccVJbEQ4zCYYhhi1YqeRc5m5+J+hPJV6bXYW6kqHHCjIzll2JOiZtnVPZDTrzOKnUHBsbQrXVAP4d8LyF7MdCVFV/UsUGaFxBCl4x9O5NJR+JiaCif7J0kmFIlSmPA1bMQWHOQYuQlItkbyEAggw7DXnNZXOfS0zEoUVrg8ttgUzTnv/FRtDLSy8N7k0uyQERUYqgzKCVfW0MAzkz7ZHQIispNfDDXqjGTgr79gd2CAN8GNyWj4irU71wCcHuG95BL+kLQSV7MjwsHZyHB6jWSz/0qoEzex3L85FzBSChLJYomxHRXD9obyxtHawbfBuJflTpMxnvv9g3DLBUs33F3cSVSjMufesDnE4Wlyf6h6qDeEKC7Cnya+3HbPUWJhCb6hP2j9ZGBskUU4shrGDSW+SNqIqHCCay10FfKDlBgI/xGP4zQts22eppE60itRVzH7qykAY65e73Hhx5fUnYYLAguOzD2jERTHobs1rjpu7JX+e7kwv8t9MXux8irMbva7OEOZaKUwyAwGcZpVR9tr7p1W4soqEYmw0i/CZovyEpRQgNMdiWV+Y+8yGafJ7EVE0cJkGhknJETm8Qq4UiQuNYOJB86TF6yDF/RCqA8wZTBqKICn7FlnMeBPZJokJ5hpwmcYIClF/qTDVII6GDsyLsRIikgjpYLmCiW+w6WZCWDReCtXa/qJw5MJo90oKiYQ0Uitib0VAqX4hDdFUA3OSiCAEK38kPOmBz2x+uUiT8nJotMHycUyY4DYmlDPeworBU5hpaaZyCRna5LwrZoSX8+OVovo/q032VkkYvq0FeUkgHqcnM10Ekb4ZM2BLo7Y0xNahiSS4XtDJBvUw+9EnrWgF1EePHWynfzIXsYLe7kWgAEB31xSjt71lRnO1q5GteV/+vLej4gV1puuOe/tFabNmtZ0E9s88eNNdeaHRjQR5TI8s4Fps+YCl/PNZXza3x46i8W7aude4kfJgufQOQv21/m59MCGVAyX+PlXcIVXiCWM34pGT1w9n5qF0yY/owMv54glexVTRCHNgCStffAUyBAmYMgRufx8/NOHL8fR8mqV4MJ7hBiTUPUzCYwFZoMBbTGqq1RXYI5jAX9Oc+xz+B4Tv/25glM9i0Qz/txYjjb6tRajaTOE8Y6kbxOEzHtGWp3QxLe3xlAG+0NeOlQDyX+hGW709gm5e8KVKTek7zefv7w7FiROW1zEEE+1zeEkfbHQAVXoEW3eQnwK7eH3t2EydxuNIMFdgd3DWzkS2YAGQJzKR3VvbkO+7W0IiU704YoyvqMbHUlmqEAIygPAzLQGttzpHCOd0rTIy+X5NqsvgH+zDq6NTFeQX9aYKYOI5Z0Ys9Sfev+vwoJQnBoh3J5ARwODVR18O07EIVlklhst2riyE6ssrlAjwhYme+ZSqHybNgb9/gtj4OVkSr/iZYr7DIEPNTLUl0iK4WVN2vkt1FNbgzEG/MNWB//rL3Mf1YqwS2iY4o3QPfRW1DRfL6r2fPiS/qF3Qhf2lZKJcR4voXniPyFQh1TEsgXT6zjPV5tnemJG95lLQ8vESeLIFBJT8i9MoPdfHla2mXKgeUTNeW+qpfQL7pjISNQP5ZFln3J1/NOixmqt3iKHh1VhYiVY3fLYW29VF+uW82Npf1i+0+tcT8O5tId9laHIy0cwO2hUQfFQGsVjedizF2I7iQJD0rEI9SbBLHoajlUPwNp+No2OhIch+Cp4Go3WoO0MTryPH0rkPJPoO+8XgnDlgYKn8PpcCk9qldKBeDe3rI6pubtzJk+ddYwuEVOxQcYat/IRm78TSiqXlrLA0cn0lH0OvAotVlfnbIqiwx0casujUAzORSASW4SWE+Zn5KNTShqS1U3Jxgeq0cwnEsbJ4SnPALdwOrIohNo2Wfs3PMLbyeKGGridLG+o7dvJ+gbVM1vIt76zEUNqGL5aJLKtKZdjiHKVAcyDxTs8ZYr3RGr9M0pZChNW6ciRfGPMvEN6qECILUFQueKKTOBIxhuxThIjJLLn6AjNe4Ntt995/4skj6JxmNrdbnPt0eYZ+AsyQx2lTWmBApt4ULa3ZQaZvWAg82xzNI2lkrJFmq2gvoENB4QeGExh0J5j/OuixWyW5KTWu18Nccg7djVWhT8MV8JfBfwKQGl47rLOnQOCK4aR0++8N5d5iV0fo4Y9mnpgmjZ4HPA1pYvLQwoZebxrLmvxnLN40NbyHER5mWDSnhhpdJFfJ5tsu1hlXj2v70N4jhArUSEv0/pFuQZfTjeEkIvOpDFvH4pAR7INNxOTSBhuLL+HsSJ1HE7eGEn1ea53DM1DS+7iAmlRdsN1cvyJHjjAwwgebBAnoeh+U0qNzsjBqZEq+i45tpbmz1Ay3hAibm2tEJmgCoQ5Ikzj1s8fJLLm+8XLK+lbW3s6OpAzqFKgl3N1lCmwAQUV8PxFI+0e7SipaIlpRHployTNXebwt0qb/6DYOJoOlpsFOENq/0/w/3B5zbDfVRwF7ENBquLP0TRys4x6bB5mYDztyZFTm7hlMeuEglNJ5nYjwn05NcOrRrDHuaL2B3LCPc1rqDM0rrXC03b+pMUUezRym1au4KqvSKzzjL9TQTUnoGUAF7iDEqHIM2AheciNPpbX3uIa/x6LGd6dYbf/OYXLpsCpaXrTarIAzCH8DSaSpf4/cpGLKmz+l8qBlnO0briBWrcDNwHG/kfxmbFuAV0lyewP5TxZQMfVFWmrr0GcAe806uzz4jLfXL/sb/g72GUrot6i8cioB3MVz1Sh3X1M/bFwP7nVtMkzUCHJze2/SXn8cTwwUAR3UDL+Npjd1LQpCod8xYvC38BFGmLZT3j34OTDPUMbYVkbWnu8/1m+JFyCbK2TKR5+DYaLyLZs6wJsLMoYZkrig7eGiZd0No0kOqoViD2rtCPma+L7YtsgLh/jyRNiP31/wdDh15RVyceWmyDs08xrhZyhdXnD8PStyz9IjDhtzH+zJBGLpcguJoJNkWasqw0z92iKDQV+wssk7qSbEc/8oeGZkzLz8Tjiai2SNn1i5qJ0WRg0m3jQgB/GRINj6bUWAd74BLc/NwCORFvz8V78Xwr2V8Rb1+FfarXjrhDEMFRxl80UOU7Utp392Yvo6dPo6XOMAj57Fj37Hm3J58+iF8+jw9n30eGLWYQf8nr6/bPo6HAW/fk5PAMX1L2lyMWQYIYxp9PdoNaeTT5rawbU9zGEvHA3h79FDWKpL9ypkpktwds9XB0MGr06x0wLamFCY/nx0A0ym5MBlXlC5cODg9nY6Jziywdjc40q8A687cfdCP50psgQR+ZvDMPy7JKhOIbOCNpvwKopO0Di97BBrUVrfyKKkUWWZy3IUUuq03qEbUxBKbCnQmn8wa1p/w1dLHH6nAVPqA9qSQgUW3na1s6JMChASEiVKyuntmr25+r4F5+1ckqtpxSXwm0dOOoBhGCd1tAorubiOohuk2vRdEW+QPsyp1IyVCRuQZdktMDIfSQJvqXZUpyxlyvwiB3jsT837xwQ7/vi2fwwYmmK2c3019EAWIL5stXqjncXzG/ASJkLpQSXbFLBBRuQljaNLC06DJpFtk6MbCUYGSovvHUAKE7Fx2FROwrvR/HR89nsxbPvj+j8vs0mHez0nR9N0+mU/ne0LHYs/EKqbEJzLzaMig04HiMHrsBiOq/zXER66CugcpeOzWCjnqg6TuYlMAIRxZrOTGzxnBD4L9MnRzGHE9GJLQ+lJ4/2xIM9i223jTHgeiDTA9QqGTTEcg1cz6bpMAkbs9WqtiqLpd4wgZNJHa/p8wMYngX7sMTQFtKCWlcTSXv8ERbaqcmOOcGW1UXexNrRYUpFEORXWnhvMXEk+O78bSKg03xHJ3fFvvzqxhrkkmVy6/Rxx8ar0UMr2VqIaWmvwRCcGZHww6Ez7lL+MomOtws30sP6RheK3FxrUN8ANPvIMvb0xAfYtTQldqATxahYmCRP1bli9T575u5FG4Ef7e/sEy7Qw6GsMbYfJjk5mhoDX9vf+6h5FbW+/3RTAwJv43NnzZ+hMxg2kAj6lDD+aJx1oo4+ActO2tvmqyIrU9dGQGsvFZYKaGicPkTe8sjeKZBAKXGFkGwsSGzAcKaRQ2Cin4uVK9n+jyJbGl7edwwIOvA9CFuhLAUVGx2a7fe+ynqvTnBD3LJN6w5Tz/xPQurNhQCVwoANhqb3/ctIfUjW+yF5Mf1T1MvvAXHzmaQICBlePck24MFy+JDJAwbn/ZiA5GoosihHOHthnGxEUhKZ0P6ilJBHskrKk5loWtJcT3GOMYJgMpRiwdGMkhj3aGakGmCz6s7Odx3uAQNEpJdNqg8RUEM4mgKHBD1QDnojRDrtPzLXSOUzRcUy2QmXLxIjDU4fKadlz9i5dsbHtXVp8whhlQOnMODP1WWP5oZpcpHfMx79ee9BpJWDk6uRryLfwSzO0janDNicDI39B8FZjMARbhBU15psD1T47aVJ1mpbI7iY4lAS4C/19GgKKhrwixrc+kbmQSOWFmn7Op8olF3JbeIAB/R5HYuv2fGRJ8DaebY94AYm9HUR+ngR2Lvdho765EO5W9kHnWMnWmQbWHx/raNvqpMdBbaGh4v0gI+DJWgRGPAWP78H9ojc/Bx7f1PToL4LbYxjKWDcWkjkb9KJraT51xb68DArRfL5/mxMpm6mdNodnJz433kfRr4PT3EOUeeE5/s0wtCY7/4+tiTYuXc4/VPcNxUxgfiADr8kQde1TbHK1Y5Zj9Vl7GMHJqH5v3lv9OLkb95n4ku4OFaY+k0uZAJMfy3+AvefpMDlWx22xon6DUfw22Qykf/P7X9+8/V30kzbRhtCl4lhA70kdZhcst0htL42RBx8pa0FQJyjonqtNuHgvhIjatrfnIPflxAj7pXsx1dV0zT7KsMckHxTnzxCs+3R6S1g7Oby5BFLQHFPx5GSCo1mygZgqfdo8sgoomA7mD1+/HQq08fUBDmqSCCtGkpnWhUYWY+krHmk8PyIDb5HEZQMoahvEDl+dueE6PYnKWq9AKmTD+gVH3LXuxr/Cq+0RR/OmQ88/x//+AezkXFIt7DAwABzSvQ7BfZtVNA3z5JZGGEHgvZsFv705uOHT8fxFkgn5uNjKZfb/7+lH/8PMFdAwwyf4AO5CWeHO9ZTbEaGKHeYfk6P41f1WYf89BHv6mCV8+FyGBNJ01W1TFNxMHScrVbYCJUOjH0ptHMKd7BE0gWUXzQb1tH7Vb6llty4MlZnBEASd36Uyc9C4THG4EN3ue/shbNF71va3M7hqDICktxhw+u8KHNdjtZpJDYUGbuanECI7TmEF0z2ktVkNqcDbN67M6gxdbYvQh73xYnYNHNvFBo7aL4Bh/YWII1JkQstE5sNJIovsY+RiQjUfAMIxj4j3b/aBGbmKxhQyPdjjcpNU4O5OXJiT+2bGs6+yMLNkl1s8r5SZTrOJVjYl5vG4FIwWzjHhJ6sxb1iuKIXnZl35h41uUKb8fmsPxziyl3GR3X9MOUbHpt8JUD/4ekcxpWLL6FXTUEfoxXLb/Ij1z+C/6++W0z+WVnmZ/zdWs707J1jTJDYaZlBZh0R9lBSBEPD88/XamIfGvNlQEm73MXxcQeqsNwNv8h5d/yEY7K4oQp1C57sg7tPy2WuYSQ/s3/krT7v9j4gm4CJ9FBjBVugRaUqokl2QJ+eo2PPDWTpPDzodZiKR8mu4+CGJhifqCl1FrBKuBafujO7FutV2YMHAITcupskfpqieZymvt4wmgyo2NhXqg8AxGdsI5jHmT/4f4eypyk='}, 'src/reversibility/optimized.py': {'sha256': 'e6a0b7f23d42818a1672890322febe276540886aef92e3ae9c8fe2ef5e2b96c7', 'compressed': 'eJylV1uP3DQUft9fYcJLpk2zF6Q+bBmEVFhREBdB4WW1ijzJmYm1iZ21neluEf+dc3xJnJlhKSKq2o597pfvHGdZ9u3jAFr0IC3vmNVcSCF3bOC2NW/YVquPIFmvGujO3SVoxjWwUdYtlztoyizLzkQ/KG1ZraSFR9uJTTyxStft4kcpZblFbiuURIXcsJsz1NKHWz5atdO8mWhYYFayhqoR2y1oNFXwTQeer9SwB20EHkTardIfuG6qgQtdsF40gxLSVnhcw9nZWd1xY9ivMBpofgyX+YH6m6B+dX3G8PvaWG5F3YNtVeNOGthGPXltHws2FOyhYJtO1femYG3BXgxcc+RA44IU+pC2jET0/5atZ6YFVQN7UQNeD+G/pX0aYEHC+wHvvenCVGR9zY2tQFJ8nGGBd7UUTZImzh3YmdVdnWREb72lTMhg8uwWfRQBFJpGP3d0MThZTEWGvi4tMnwPlKBqw+t7F1TPkgZxYtBgRy2d0LN/SM/Xp+ol5m3S4RLXoJzmIUlR0FyhL9G0prIgjdJmIqIqESjZINXt3XT8Qdg2KWUKaRLMgoXMrEP2UDPFez0lZbUM6SLmvtBDWn0CDsjp+5y9HTX5zAb27jc2IJdQo2EPJftZAnNNQNYPLTOg92BQtGsg7DUuG/bH97+UR0ITt7wHFfmfn1DvAhh0Vg+hei2v23yFnfowCg3GMVf5ez0m5XXgtqv8K/YiNMmLgzaeCmtSdVrSdE+Nkj9MtrBXXsl08B+s+5z9ADAw2wKmxNUXGmvHAQEIg3cuGio4+8R4XY/92HEHY8bqscayhePg0ofeIBquD1HQRfkkA3357H2RevrS+7YqPoFzSGOIVe/iWs5dh5EpWB565Dge/gINdw7cXl/dHZFMnXJ7cX1xN5FeXR+TBgSZjZvSk1o5HR5bA513vfBKDhHDW1uwn5SE+PeLyT4cDYQOrWgwgxVBCpjcjb6CiQbxWdPIWN/wzsQ+FVu6KWUjevYZ1Sv2Kx2Ylg9we3nHvvKzs8TRuBW7suePlcE6o5KpOpA7284thOPVYAnxboRvtVY6z97JPe9Egx06jJY5oedhyGbe+0EZQfVl5trRNJfz1Ap02sHPmg5TWH9ELm+fVffoM/QbaBqc/8S+wjryl1FJcj/pXcU4tNxwa3UMWFYrDVkCEEji4reEDDz1Ojx2UxDnKUHR9LcRsSv66Yg0YCB8U2XHKHQcyhsHKc4CFjudJUKgmRAmWxZWVbg4LfeFkg9D95Tj+MA/wcYwxf0vY2GojPhIFZbeL1trUgTdYWTm3FAk80dPu6Q7HdNPdP6Eu4txk1p9bJo7j1aF9vIcW4GLXSWV7una9xSOuioUbh5HITZ0LWjs4CKE7Qxrh7hTXyU7D1tjwuux4ZmbUBOjK4Tt8MVVUgHBlIMR7LnjtP0TmS5fZ9eBatspbi9fY81utsnxJpz/dTspvAumVtMYp1+LGMw7cCnHrotOx0DU7SjvcZ2YVpAUXzq+wfxSt0rVYxTRjsJzVK4/8cogqoFeO+g6je6TsWsfmgVsES4a47SDbNaZ8zR2KW7ytCFQucUl7jza+QaldMCxfgBdZnsM6wZHm34Kq4Toe2gEImb3VPqdzM1IpcWOyoHVmvQiymo14GQMeUH46kT9VLLvgDfsA4hda1FRj+8Mw6yA5o0T1dLtvHDxpsEUsQmM5quSfSMMZcazOBWUIiKyysnie4WASiGhQ44vHIBA6T3hzlQfdkMIRHN+o5qnaW0sY7RiqaYpYl+u2QWxJUl0Z8/h/FsSwAgtXIFT7yDuf/SbQyqoH9HODQTY30PoXbeepT02A33SZwkY+xmHbfwvw86Lx8RvJ9p0nXMDJn91WcQ7P3FeXd49s05ZrvHFQQPL13siJ6ijUrHKJgvRR8Ck5LQGHE2y2NRJL39x5QURnKFn+CKk7dmNxQvsAJB5sGG1bK8kQJ8W0Yhc6UoQv07thPPypuzw2cx1TmG8dfZce6teLrTfxdHR9RWVb+n7YXU4L9MOdshotbBKnpiC/n18D1pimOPjeNGJlRn7IzZSQG+fQ8Lcu1TEDD7ny9Jq6E7a7cDn2Oyg/6ZcWPBftGMJ77CJoBKygcf1q8uLCyroZnRv+nWGzmSHFh5O0dOd+ru8l+qD9DYGZw5Exe5FWFl7uvO0hxe0eUKMIfLozoRxu6kzKpyV7p+EHB8s0wyZ9Set89Irn9vVtUZCS8tyjClRFonl/wdVZruo4N0TJrZ4xU3uiVaLoTmbHcfkMFZG8sG0yuZ7iv88o95hTgcMOz1x3/7yOwsv82uEE9yqmBqs6NEHan0YEMi5lMoichLEIeBHsWUC4OgIroEcl3KvrAiw896JXh1tF45ojqxVuGAMI47aGgt1PUPdSdGNqO2xyD/vrw+8Xjn8wjfunuDLqxQWetwb//pn6Z0wJ6TfnhI9y717JhD0rj0W6I7z56UuUuxOzv4GDh51mw=='}, 'src/reversibility/kernels.py': {'sha256': '9b12c5ffcb9ce986fb04c2c52aec4a7bfbe3340f7b743a4055fa2bd511c1e4dd', 'compressed': 'eJztWFtv2zYUfvev4DwMkFJZcdw0D009tMutxdImaOItRVAIjEg5XCRRIynb+fc7vOjmKE3cbQ8DFiCRRZ7znSt5Pmc4HB6tCipYRnOFU3S5i+6oyGkqQ3QwO3yH5lhRibJSKlRgKdENTbigCFTgmeE8pihOMctkOBgcHIEyLSRStxThkjBFCXqP02SU8rnkicrwCo1+RsfnLyfo0+kpKnjK4vsAsTxOQTyfG+mB4GVu3pwxuipABN3g+G6JBQnRJUsBWVBSxorxXCK84IwgjF6N70ZLRig6uHwXDt4VheArlkEIGmO7VkBcECoQk0jSAgvYT+/RAqeMwEcSoJwrdMPUkkmIVPAMniQcDIfDAcsKLhRS9wWV9QsX8e0gATn7McSl4nOBSZiUubXnJDmkKyIsSaiAdDN8k9IaRDDF8+5bmOJ8XuI5JBNymg4Gg7du4w+mBoQmKAI8CACgUuldBej8I/xeBOjg7PTiNaiEMWRHQewiQJcfTo8eLP5yenbwa3fRfz1A8CP4MkAKEo2mehvSACFlESPe2A/WVnZ8oxLzVAtrnS2LjF5oSchwPqfe2JmzwiuLm3JMvCuQA3ugpR2HF0AKDNwbsxIgDi0lpqMExJU3ZHky9P1QcQ8QzNrLiUXNLCo0mrcK0NguSrsoy0zLQ4wetCHK/FpAb0LpqHf+sfbEpEv7D+EEILwmePGIoPQfKRMImxJ9aar0GZ6nh/B7drFJzT4fHc4Ojh4rWk+5rOt1KZtyWKRO5kw9Hk2DAXljF/uL0k64xXosU31Y49A5g+HYlq1iZnWtCM3hQOYYylBZmXuuuhKMuApnusIWxdTZJQGLOVUt775Y5+yuuQH0ppX6YYpGO+Px090KG0vwnnoGIHD6xmpfj4LDBcB5tgudj/CxFZnRuwqJvmdCmlJ9O0fqvheu7snPrmpBDdrdPz2s9tuW1mSgFSup9bD0LV5AVKH/WI/rS8/1eNXbJwE6vOpv7v/W3QNhP1nMCrOqhO+3Fqv0+xsV95sN+yM6E2wOhUzNOF1CRWBY5xSGNltQVBZSCYozO3B1cRiYCvR4zlGMpYKpws3IDQ1crQjxjCqDJxv5S1xqbeOYokzr81Dhmx6ymdFHVafQh/xX2z1A3RMJDbwyGN3ePewvpZZtyqlbFwgLMBlDIg4El/IoV4IX997a9D5209s15VupwL04o+qWE3cVJQgIiuYkXqzADETClIQnvgEG5fT0D0vcXkjogsU01LnU4QzjkuAh8JF63+xoAgKMx3lkMryzF6DWKyS8gTdnBmu28htOS3okBBfe8NIcUASsTNA/SyagNQylOz7f2ds2LWFt7oM5JCEbuUIJTlPNs4Z+j+9AyjLt9ET7q11060xGcIIVm5e8lJ5vojEZcNFqnU74bYkqEzY2lqu93dZ2XmY0BcgGQd7igl6Pvz4V/UHtEbpW/I7mMljwGN98dUAI5wQuShXfaqZpEmONW8tVzkgrE25nWjnXDroW0nPeNJwRbPm8j25SHt/B6s54srtv7iozKO09GhO20AcGlI1cg1iYOWjSQ7NC3XuetWEAfH2b65xOOxmGVZ3YabdjGkjZhYxSdke9ImuFsW7U2vwOWynpsyUyfx/SIyXtdUW0XGmT3OtO7F+96sQVQBAKl3e36/II106ZRXBGCzndDRDNNeuOkiJKSgmHe3oMqLRrTJtwhlombNEbUwKeKQlcEGumXVFzulJRwZdURDyJJl5Vso1cgqsllHgBElxE1ZegB25Zd1oFpKoUufPOsCO4+vrvsbd930uqC642aG64aoi0Lp9+R6Cold8kUnD6uJBPn5Fawl7/611hRbvF0h7VXfHgKOmj5vcUsSpeFU/nLNczwwFsWC+XeY3xiecUBo7OY6wnTUTtqIl0PXqnhVN+MJxCXBTp/bpOHw+TSzZPy6gaTLMAnYEj6+Tr9w+Hl++fz8hYDxN7BtG6wdKwCYa2t61JreVN4I97ewHQP9kXo6H/39CiOzMQ0CABiL1Bn54gYwt9//dow8NYeB4KjMJSO21c2UbeTmgj1FxlpBd9S4lmG/LzMx1sYOG3rK/OoW/VsT5/M0umZ/8Xc5Ni1hy4ATqxdXheK8wzbr4TQhP0t8LDltmq1DbpEjKPgH5QkVHCbNa82nXXLJvBAcI6JhSrCmjLxeIc1i+j2m1/nVg3hSPzumX7RZrqtNzX6Wm1uuXfx6Wk5GLJTk5nf595l0WXbZfF40xb7z2fZVdcF7R6eC6sWkI62vkKrT95ipWaoJGNuqHlDeo6Q9cklS7g61pivuUtGVG3LUZq3qHSHTfgdE5qCV6qolTrBHKrVngNGoHFaahknb2K2lUpa43e7pi59tqz1xqtyHuAJq/2fE2lyiJwDlXPRsa4YET/EXoEDbE+kK3Ff4cE6ci+xXgqFRDqpRwN8Sn7iE87mvXB0E19jd6b/OZ/AKRskZ9att1H29uT7ylHxX5Kx3ucvHXayzihqctb+xBnKXi3aiXUwcB6SPgy91q3haNCeqssvFV1XQGU+4YFZ9rYCc2rbEDNe6gVnV2da/2v/PCjaYRL+Oy5raCR9gd/AdMn3Aw='}}
for relative, entry in EMBEDDED.items():
    payload = zlib.decompress(base64.b64decode(entry['compressed']))
    if hashlib.sha256(payload).hexdigest() != entry['sha256']:
        raise RuntimeError('Embedded checksum mismatch: ' + relative)
    destination = REPO / relative
    if destination.exists():
        raise RuntimeError('Refusing to replace existing source: ' + relative)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(payload)
BENCHMARK_SCRIPT = REPO / 'scripts/benchmark_optimized.py'
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
RECEIPTS = OUTPUT.parent / (RUN_ID + '-receipts.json')
try:
    triton_version = importlib.metadata.version('triton')
except importlib.metadata.PackageNotFoundError:
    triton_version = None  # Kernel gates will record an explicit failure; Torch paths remain available.
RECEIPTS.write_text(json.dumps({'source_commit': PINNED_COMMIT,
    'embedded_sha256': {p: v['sha256'] for p, v in EMBEDDED.items()},
    'runtime': actual, 'triton': triton_version, 'declared_environment_differences': mismatches,
    'data': receipt, 'probe_only': True, 'fixed_reserved_headroom': .10,
    'gradient_accumulation_steps': 1, 'validation_loss_ceiling': 5.56229485,
    'optimization_priority': 'numerical correctness, memory capacity, full-run quality; speed optional'}, indent=2) + '\n')
print('Capacity harness staged:', BENCHMARK_SCRIPT)


## Run numerical gates and capacity experiments

Fresh processes isolate each probe and allocator setting. Regular probes:5 warmup and20 timed complete updates. Capacity discovery:3 warmup and3 timed updates; selected maximum:three regular-duration confirmations, including checkpoint overhead on the last. OOM and candidate failures are recorded and retained. A numerical failure excludes the affected configuration; if no shared chunk passes, optimized probes stop.

Both baseline and midpoint receive shared optimization and kernel tests at matched physical29 and50. The lowest-reserved-memory feasible configuration is selected at the largest tested fitting shape; capacity growth then uses that fixed configuration. Allocator `expandable_segments` is separately tested at50 if a candidate fits. This bounded search reports the best verified result among these paths, not proof that every possible implementation has been exhausted.

CPU snapshots preserve independent model/optimizer/scaler/RNG/cursor copies. Copy-to-CPU and disk-write time are recorded separately in `checkpoint-overhead.json`, outside training timing. Probe snapshots are not production resumes. GPU busy sampling is coarse device utilization, not SM occupancy.


In [ ]:
WARMUP_UPDATES = 5
TIMED_UPDATES = 20
command = [sys.executable, "-u", str(BENCHMARK_SCRIPT), "--suite", "--config", str(reference_config),
           "--data-dir", str(LOCAL_DATA), "--output", str(OUTPUT),
           "--warmup", str(WARMUP_UPDATES), "--updates", str(TIMED_UPDATES)]
with (OUTPUT.parent / (RUN_ID + "-orchestration.log")).open("w") as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    returncode = process.wait()
print("Exit code:", returncode)
print("Evidence:", OUTPUT)
if returncode:
    print("One or more passes failed. Continue to the next cell to inspect preserved logs and results.")


In [ ]:
from IPython.display import display, Markdown
report = OUTPUT / 'REPORT.md'
if report.exists():
    display(Markdown(report.read_text()))
suite_path = OUTPUT / 'suite.json'
if suite_path.exists():
    suite = json.loads(suite_path.read_text())
    print(json.dumps({'batch50_decisions': suite.get('decisions', {}),
                      'capacity_search': suite.get('capacity_search', {})}, indent=2))
    for row in suite['records']:
        if row['result']['status'] != 'complete':
            print('FAILED:', row['name'], row['result'].get('error'))
            log = OUTPUT / (row['name'] + '.log')
            if log.exists(): print(log.read_text()[-4000:])
print('Full50M training has not run. Final loss limit: 5.56229485.')
print('CPU probe checkpoint binaries remain separately in Drive; do not use them as full-run resumes.')


## Export and next decision

Export includes all numerical failures, memory/timing results, source/runtime/data receipts, utilization samples and checkpoint-overhead records. Large CPU checkpoint binaries stay in Drive; their hashes/sizes/paths are recorded. Return this compact ZIP for review.

A capacity claim requires a numerical pass, physical==effective batch, accumulation1, startup/training/checkpoint reserved≤90%, and three fresh confirmations. A first failing boundary supports a maximum for the selected path; if the ceiling fits or confirmations disagree, capacity is unresolved. Do not use the fastest one-off result as proof.

After capacity verification, run the highest confirmed midpoint batch for exactly50M valid training targets and evaluate the same1M-target validation stream. Require loss≤5.56229485. Keep physical29/accumulation1 as the same-session baseline control and optimize ordinary baseline too before attributing shared-kernel gains to reversibility. Batch changes alter optimizer update counts despite a common target-indexed schedule; report this explicitly. Calculate savings only from equal-quality end-to-end GPU time including evaluation and checkpoint stalls. Speed tuning is optional after memory and loss acceptance.


In [ ]:
import zipfile
BUNDLE = OUTPUT.parent / (RUN_ID + '-capacity-v2.zip')
if BUNDLE.exists():
    raise RuntimeError('Export already exists; preserved evidence')
with zipfile.ZipFile(BUNDLE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    if OUTPUT.exists():
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix != '.pt':
                archive.write(path, arcname=str(path.relative_to(OUTPUT.parent)))
    archive.write(RECEIPTS, arcname=RECEIPTS.name)
    log = OUTPUT.parent / (RUN_ID + '-orchestration.log')
    if log.exists(): archive.write(log, arcname=log.name)
print('Evidence:', BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
